In [1]:
! pip install --upgrade pip
!pip install kagglehub
!pip install Pillow

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0
  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.18-py3-none-any.whl.metadata (13 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.18-py3-none-any.whl (201 kB)
Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl (173 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kagglehub]━ 2/4 [kagglesdk]


In [1]:
import kagglehub
from PIL import Image, ImageOps, ImageEnhance
import os, random

/Users/bibo/dev/MSc/HardSofCod/EmotionDetector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Download latest version
path = kagglehub.dataset_download("minhtmnguyntrn/affectnet-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/bibo/.cache/kagglehub/datasets/minhtmnguyntrn/affectnet-dataset/versions/1


In [3]:
path = "/Users/bibo/.cache/kagglehub/datasets/minhtmnguyntrn/affectnet-dataset/versions/1"

In [4]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
data_path = os.path.join(path, "data")  # path from kagglehub

TARGET_SIZE = 160
IMAGES_PER_CLASS = 2000
DROP_CLASS = '7'  # Contempt

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

label_map = {
    '0': 'angry',
    '1': 'disgust',
    '2': 'fear',
    '3': 'happy',
    '4': 'sad',
    '5': 'surprise',
    '6': 'neutral'
}

output_path = os.path.join(NOTEBOOK_DIR, "emotion_dataset")
os.makedirs(output_path, exist_ok=True)
print("Output path:", output_path)

Output path: /Users/bibo/dev/MSc/HardSofCod/EmotionDetector/emotion_dataset


In [5]:
valid_images_per_class = {}

for class_folder in sorted(os.listdir(data_path)):
    if class_folder == DROP_CLASS:
        print(f"Skipping class {class_folder} (Contempt)")
        continue

    class_path = os.path.join(data_path, class_folder)
    if not os.path.isdir(class_path):
        continue

    valid = []
    for img_file in os.listdir(class_path):
        img_full_path = os.path.join(class_path, img_file)
        try:
            with Image.open(img_full_path) as img:
                w, h = img.size
                if w >= TARGET_SIZE and h >= TARGET_SIZE and w == h:
                    valid.append(img_file)
        except Exception:
            pass

    valid_images_per_class[class_folder] = valid
    print(f"Class {class_folder} ({label_map[class_folder]}): {len(valid)} valid images")

Class 0 (angry): 5418 valid images
Class 1 (disgust): 4221 valid images
Class 2 (fear): 5388 valid images
Class 3 (happy): 5431 valid images
Class 4 (sad): 5422 valid images
Class 5 (surprise): 5430 valid images
Class 6 (neutral): 5414 valid images
Skipping class 7 (Contempt)


In [31]:
selected_per_class = {}

for class_folder, valid in valid_images_per_class.items():
    if len(valid) < IMAGES_PER_CLASS:
        print(f"Class {class_folder} ({label_map[class_folder]}): only {len(valid)} available, taking all")
        selected_per_class[class_folder] = valid
    else:
        selected_per_class[class_folder] = random.sample(valid, IMAGES_PER_CLASS)
        print(f"Class {class_folder} ({label_map[class_folder]}): sampled {IMAGES_PER_CLASS} images")

Class 0 (angry): sampled 2000 images
Class 1 (disgust): sampled 2000 images
Class 2 (fear): sampled 2000 images
Class 3 (happy): sampled 2000 images
Class 4 (sad): sampled 2000 images
Class 5 (surprise): sampled 2000 images
Class 6 (neutral): sampled 2000 images


In [32]:
splits_per_class = {}

for class_folder, selected in selected_per_class.items():
    random.shuffle(selected)
    
    n = len(selected)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    # test gets the remainder to ensure no images are lost
    
    splits_per_class[class_folder] = {
        'train': selected[:n_train],
        'val':   selected[n_train:n_train + n_val],
        'test':  selected[n_train + n_val:]
    }
    
    print(f"Class {label_map[class_folder]}: "
          f"train={len(splits_per_class[class_folder]['train'])} "
          f"val={len(splits_per_class[class_folder]['val'])} "
          f"test={len(splits_per_class[class_folder]['test'])}")

Class angry: train=1400 val=300 test=300
Class disgust: train=1400 val=300 test=300
Class fear: train=1400 val=300 test=300
Class happy: train=1400 val=300 test=300
Class sad: train=1400 val=300 test=300
Class surprise: train=1400 val=300 test=300
Class neutral: train=1400 val=300 test=300


In [33]:
for class_folder, splits in splits_per_class.items():
    emotion_name = label_map[class_folder]
    class_path = os.path.join(data_path, class_folder)

    for split_name, files in splits.items():
        out_dir = os.path.join(output_path, split_name, emotion_name)
        os.makedirs(out_dir, exist_ok=True)

        saved = 0
        for img_file in files:
            img_full_path = os.path.join(class_path, img_file)
            out_img_path  = os.path.join(out_dir, img_file)
            try:
                with Image.open(img_full_path) as img:
                    img = img.convert('RGB')
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(out_img_path, 'JPEG', quality=95)
                    saved += 1
            except Exception as e:
                print(f"Error on {img_file}: {e}")

        print(f" {split_name}/{emotion_name}: saved {saved} images")

 train/angry: saved 1400 images
 val/angry: saved 300 images
 test/angry: saved 300 images
 train/disgust: saved 1400 images
 val/disgust: saved 300 images
 test/disgust: saved 300 images
 train/fear: saved 1400 images
 val/fear: saved 300 images
 test/fear: saved 300 images
 train/happy: saved 1400 images
 val/happy: saved 300 images
 test/happy: saved 300 images
 train/sad: saved 1400 images
 val/sad: saved 300 images
 test/sad: saved 300 images
 train/surprise: saved 1400 images
 val/surprise: saved 300 images
 test/surprise: saved 300 images
 train/neutral: saved 1400 images
 val/neutral: saved 300 images
 test/neutral: saved 300 images


In [37]:
print("Final dataset summary:")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(output_path, split)
    print(f"\n  {split}/")
    split_total = 0
    for emotion in sorted(os.listdir(split_path)):
        emotion_path = os.path.join(split_path, emotion)
        if os.path.isdir(emotion_path):
            count = len(os.listdir(emotion_path))
            split_total += count
            print(f"    {emotion}: {count} images")
    print(f"    subtotal: {split_total}")

Final dataset summary:

  train/
    angry: 4200 images
    disgust: 4200 images
    fear: 4200 images
    happy: 4200 images
    neutral: 4200 images
    sad: 4200 images
    surprise: 4200 images
    subtotal: 29400

  val/
    angry: 300 images
    disgust: 300 images
    fear: 300 images
    happy: 300 images
    neutral: 300 images
    sad: 300 images
    surprise: 300 images
    subtotal: 2100

  test/
    angry: 300 images
    disgust: 300 images
    fear: 300 images
    happy: 300 images
    neutral: 300 images
    sad: 300 images
    surprise: 300 images
    subtotal: 2100
